In [ ]:
!pip install pymupdf opencv-python pillow matplotlib

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import fitz

pdf_path = "2015개정경제수학-광주교육청.pdf"

doc = fitz.open(pdf_path)

output_dir = "outputs/pages"
os.makedirs(output_dir, exist_ok=True)

for page_index in range(len(doc)):
    page = doc[page_index]

    matrix = fitz.Matrix(2, 2)
    pix = page.get_pixmap(matrix=matrix)

    page_number = page_index + 1
    image_path = f"{output_dir}/page_{page_number:03}.png"

    pix.save(image_path)

    print(f"{page_number}/{len(doc)} 페이지 저장 완료: {image_path}")

print("전체 페이지 이미지 변환 완료")

In [ ]:
import os

page_files = sorted(os.listdir("outputs/pages"))

print("저장된 페이지 이미지 개수:", len(page_files))
print("처음 5개:", page_files[:5])
print("마지막 5개:", page_files[-5:])

In [ ]:
layout_output_dir = "outputs/layout_results"
os.makedirs(layout_output_dir, exist_ok=True)

print("문서 구조화 결과 저장 폴더:", layout_output_dir)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

image_path = "outputs/pages/page_008.png"

image = cv2.imread(image_path)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# 글자와 선을 검은색 덩어리로 만들기
_, binary = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)

# 글자와 표 선을 하나의 블록처럼 묶기
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 10))
dilated = cv2.dilate(binary, kernel, iterations=2)

# 외곽선 찾기
contours, _ = cv2.findContours(
    dilated,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

blocks = []

for contour in contours:
    x, y, w, h = cv2.boundingRect(contour)

    # 너무 작은 잡음 제거
    if w > 50 and h > 30:
        blocks.append({
            "bbox": [x, y, x + w, y + h],
            "width": w,
            "height": h
        })

# 위에서 아래, 왼쪽에서 오른쪽 순서로 정렬
blocks = sorted(blocks, key=lambda b: (b["bbox"][1], b["bbox"][0]))

print("감지된 블록 수:", len(blocks))

for i, block in enumerate(blocks):
    print(i + 1, block)

In [ ]:
image_draw = image.copy()

for i, block in enumerate(blocks):
    x1, y1, x2, y2 = block["bbox"]

    cv2.rectangle(image_draw, (x1, y1), (x2, y2), (0, 0, 255), 3)
    cv2.putText(
        image_draw,
        str(i + 1),
        (x1, y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

plt.figure(figsize=(10, 14))
plt.imshow(cv2.cvtColor(image_draw, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
def classify_block(block, page_width, page_height):
    x1, y1, x2, y2 = block["bbox"]
    w = x2 - x1
    h = y2 - y1

    # 상단에 있는 비교적 큰 영역
    if y1 < page_height * 0.2 and h < page_height * 0.15:
        return "title_or_header"

    # 넓고 큰 영역은 표 또는 그림 가능성
    if w > page_width * 0.5 and h > page_height * 0.15:
        return "table_or_figure"

    # 하단 작은 영역
    if y1 > page_height * 0.9:
        return "footer"

    return "text"

In [ ]:
import json

page_number = 8
image_path = f"outputs/pages/page_{page_number:03}.png"

image = cv2.imread(image_path)
page_height, page_width = image.shape[:2]

structured_blocks = []

for idx, block in enumerate(blocks):
    block_type = classify_block(block, page_width, page_height)

    structured_blocks.append({
        "block_id": f"p{page_number:03}_b{idx + 1:03}",
        "type": block_type,
        "bbox": block["bbox"]
    })

page_result = {
    "page_id": page_number,
    "image_path": image_path,
    "page_width": page_width,
    "page_height": page_height,
    "blocks": structured_blocks
}

save_path = f"outputs/layout_results/page_{page_number:03}_layout.json"

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(page_result, f, ensure_ascii=False, indent=2)

print("저장 완료:", save_path)

In [ ]:
with open("outputs/layout_results/page_008_layout.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, ensure_ascii=False, indent=2))

## 4단계 결과 정리

본 단계에서는 페이지 이미지를 기반으로 문서 구조화 작업을 수행하였다.

OpenCV를 이용하여 페이지 이미지를 흑백 변환하고, 이진화 및 dilation 처리를 통해 글자와 표 선을 블록 단위로 묶었다. 이후 contour detection을 적용하여 문서 요소의 후보 영역을 탐지하였다.

각 블록에는 block_id를 부여하였고, bbox 좌표를 [x1, y1, x2, y2] 형식으로 저장하였다. 또한 블록의 위치와 크기를 기준으로 title_or_header, text, table_or_figure, footer 등의 임시 유형을 부여하였다.

8페이지 구조화 결과, 총 7개의 블록이 감지되었다.
상단 제목 영역, 본문 영역, 표 영역, 하단 footer 영역이 bbox 좌표와 함께 JSON 형태로 저장되었다.

특히 큰 표 영역이 table_or_figure 블록으로 분리되어,
이전 OCR 단계에서 표 구조가 깨지는 문제를 보완할 수 있는 기반을 마련하였다.

다만 일부 본문 영역에서 중복 bbox가 생성되는 문제가 확인되었으며,
이후 단계에서 중복 블록 제거 및 구조 분리 정확도 평가를 통해 보완할 필요가 있다.